# CNC

## Recearch: How does dxf work?
DXF files exist of a few sections, each section is divided into a header and a body. The header contains information about the section, while the body contains the actual data.
In each section you have elements of types:
- LINE
- CIRCLE
- ARC
- ELLIPSE
- POLYLINE
- TEXT
- POINT


Lets try and open a DXF file with Python.


In [ ]:
import ezdxf

file = "TestDXF.dxf"

doc = ezdxf.readfile(file)

# Print the elements in the modelspace
for entity in doc.modelspace().query('*'):
    if entity.dxftype() == 'LINE':
        print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
    elif entity.dxftype() == 'CIRCLE':
        print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
    elif entity.dxftype() == 'ARC':
        print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
    else:
        print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")
        

Line from (-2.30091294273734, 2.31162216514349, 0.0) to (2.699087057262659, 2.31162216514349, 0.0)
Arc at (0.0, 0.0, 0.0) with radius 1.792858117766655, start angle 255.553015844033, end angle 14.46433084498309
Circle at (0.0, 0.0, 0.0) with radius 1.0


Top! We kunnen de data uit een DXF bestand lezen met de `ezdxf` library. Wel zie ik dat (0,0,0) de oorsprong is van het bestand. Dit is niet handig, want we willen dat de oorsprong in het midden van de tekening zit. We kunnen dit oplossen door de tekening te verschuiven naar het midden.


In [3]:
import ezdxf

notCenteredFile = "NotCenterdDxf.dxf"

def center_dxf(filename):
    doc = ezdxf.readfile(filename)
    msp = doc.modelspace()
    
    # Find the bounding box of all entities
    min_x = min_y = float('inf')
    max_x = max_y = float('-inf')
    
    for entity in msp:
        if entity.dxftype() in {'LINE', 'CIRCLE', 'ARC', 'ELLIPSE', 'POLYLINE', 'TEXT', 'POINT'}:
            bbox = entity.bounding_box()
            if bbox:
                min_x = min(min_x, bbox.extmin.x)
                min_y = min(min_y, bbox.extmin.y)
                max_x = max(max_x, bbox.extmax.x)
                max_y = max(max_y, bbox.extmax.y)

    # Calculate the center
    center_x = (min_x + max_x) / 2
    center_y = (min_y + max_y) / 2
    
    # Shift all entities to center them
    for entity in msp:
        if entity.dxftype() in {'LINE', 'CIRCLE', 'ARC', 'ELLIPSE', 'POLYLINE', 'TEXT', 'POINT'}:
            entity.translate(-center_x, -center_y)

    # Save the modified DXF file
    doc.saveas("centered_" + filename)
    
    
# print DXF file
doc = ezdxf.readfile(notCenteredFile)
for entity in doc.modelspace().query('*'):
    if entity.dxftype() == 'LINE':
        print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
    elif entity.dxftype() == 'CIRCLE':
        print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
    elif entity.dxftype() == 'ARC':
        print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
    else:
        print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")
        
# Center the DXF file
center_dxf(notCenteredFile)

# Print the centered DXF file
doc = ezdxf.readfile("centered_" + notCenteredFile)
for entity in doc.modelspace().query('*'):
    if entity.dxftype() == 'LINE':
        print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
    elif entity.dxftype() == 'CIRCLE':
        print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
    elif entity.dxftype() == 'ARC':
        print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
    else:
        print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")

Circle at (-3.127090446650982, 2.263758471235633, 0.0) with radius 1.018999695069109
Line from (-4.732520133256912, 0.9840681217610836, 0.0) to (-1.521660760045052, 0.9840681217610836, 0.0)
Line from (-4.732520133256912, 3.543448820710182, 0.0) to (-1.521660760045052, 3.543448820710182, 0.0)
Line from (-4.732520133256912, 3.543448820710182, 0.0) to (-4.732520133256912, 0.9840681217610836, 0.0)
Line from (-1.521660760045052, 3.543448820710182, 0.0) to (-1.521660760045052, 0.9840681217610836, 0.0)
Line from (-4.732520133256912, 3.543448820710182, 0.0) to (-1.521660760045052, 0.9840681217610839, 0.0)
Line from (-4.732520133256912, 0.9840681217610839, 0.0) to (-1.521660760045052, 3.543448820710182, 0.0)


AttributeError: 'Circle' object has no attribute 'bounding_box'